<a href="https://colab.research.google.com/github/chuy-zip/PROYECTO2_DS/blob/main/PROY2_DS_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!git clone https://github.com/chuy-zip/PROYECTO2_DS.git

fatal: destination path 'PROYECTO2_DS' already exists and is not an empty directory.


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import pandas as pd
import numpy as np
import torch.optim as optim
import time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [16]:
df = pd.read_csv("PROYECTO2_DS/data/train_clean.csv")
print(df.head())

print("\n CARGADO DE DATOS ")
df = pd.read_csv("PROYECTO2_DS/data/train_clean.csv")
print(f"Dimensiones del dataset: {df.shape}")

# Ver distribución de clases
print("\n Distribución de clases:")
print(df["discourse_effectiveness"].value_counts())

   Unnamed: 0  discourse_id      essay_id discourse_type  \
0           0  0013cc385424  007ACE74B050           Lead   
1           1  9704a709b505  007ACE74B050       Position   
2           2  c22adee811b6  007ACE74B050          Claim   
3           3  a10d361e54e4  007ACE74B050       Evidence   
4           4  db3e453ec4e2  007ACE74B050   Counterclaim   

  discourse_effectiveness                                         text_clean  
0                Adequate  hi isaac going writing face mar natural landfo...  
1                Adequate  perspective think face natural landform dont t...  
2                Adequate  think face natural landform no life mar descov...  
3                Adequate  life mar would know reason think natural landf...  
4                Adequate  people thought face formed alieans thought lif...  

 CARGADO DE DATOS 
Dimensiones del dataset: (36765, 6)

 Distribución de clases:
discourse_effectiveness
Adequate       20977
Effective       9326
Ineffective     6

In [17]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['discourse_effectiveness'])
print(f"\nMapping de clases: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


Mapping de clases: {'Adequate': np.int64(0), 'Effective': np.int64(1), 'Ineffective': np.int64(2)}


In [18]:
print("\n PREPROCESAMIENTO")

# Tokenizador
def simple_tokenizer(text):
    text = str(text).lower()
    tokens = text.split()
    return tokens

# Aplicar tokenización
df['tokens'] = df['text_clean'].apply(simple_tokenizer)

# Construir vocabulario
def build_vocab(token_lists, min_freq=3, max_vocab_size=30000):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Ordenar por frecuencia y limitar tamaño
    most_common = counter.most_common(max_vocab_size-2)

    vocab = {'<pad>': 0, '<unk>': 1}
    for idx, (word, count) in enumerate(most_common):
        if count >= min_freq:
            vocab[word] = idx + 2

    print(f"Tamaño del vocabulario: {len(vocab)}")
    return vocab

vocab = build_vocab(df['tokens'])
vocab_size = len(vocab)

# Convertir a índices y aplicar padding
def tokens_to_indices(tokens_list, vocab, max_length=150):
    sequences = []
    for tokens in tokens_list:
        indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
        indices = indices[:max_length]
        sequences.append(torch.tensor(indices, dtype=torch.long))

    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=vocab['<pad>'])
    return padded_sequences

# Convertir textos a tensores
X = tokens_to_indices(df['tokens'], vocab, max_length=150)
y = torch.tensor(df['label'].values, dtype=torch.long)

print(f"Forma de X: {X.shape}")
print(f"Forma de y: {y.shape}")


 PREPROCESAMIENTO
Tamaño del vocabulario: 8457
Forma de X: torch.Size([36765, 150])
Forma de y: torch.Size([36765])
